# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitshank-2338/flyrank-ai-starter-notebook/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Use a shallow decision tree as an interpretable comparator to the frozen stale-and-visible rule. Both rank the same held-out clients and are scored by precision@50 and base rate. Grouping by client tests transfer to clients excluded from fitting; it does not provide temporal validation. The starter snapshot target is contemporaneous, so the result is an exploratory association, not a forecast or a causal claim. `trend_pct`, `trend_direction`, identifiers and product flags never enter the features.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import confusion_matrix
url = 'https://raw.githubusercontent.com/Bitshank-2338/flyrank-ai-starter-notebook/main/data/raw/content_refresh_anonymized.csv'
df = pd.read_csv(url)
y = df['trend_direction'].eq('down').astype(int)
features = ['impressions_90d','days_since_last_update','avg_position','ctr','content_age_days']
X = df[features].copy()
X.loc[X.avg_position == 0, 'avg_position'] = np.nan
train, test = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42).split(X,y,groups=df.client_id))
assert set(df.client_id.iloc[train]).isdisjoint(set(df.client_id.iloc[test]))
model = make_pipeline(SimpleImputer(strategy='median', add_indicator=True), DecisionTreeClassifier(max_depth=3, min_samples_leaf=50, random_state=42))
model.fit(X.iloc[train], y.iloc[train])
score = model.predict_proba(X.iloc[test])[:,1]
baseline = (df.days_since_last_update.iloc[test].ge(180) & df.impressions_90d.iloc[test].ge(500)) * df.impressions_90d.iloc[test]
def p_at_k(s, target, k=50):
    return float(np.asarray(target)[np.argsort(-np.asarray(s), kind='stable')[:k]].mean())
print('Test rows:',len(test), 'test clients:',df.client_id.iloc[test].nunique(), 'base rate:',round(y.iloc[test].mean(),3))
print('Frozen rule precision@50:',round(p_at_k(baseline,y.iloc[test]),3))
print('Tree precision@50:',round(p_at_k(score,y.iloc[test]),3))
print('Confusion matrix, threshold .5:', confusion_matrix(y.iloc[test],score>=.5).tolist())
print('Feature importance:', dict(zip(model[0].get_feature_names_out(features), np.round(model[1].feature_importances_,3))))

Test rows: 7115 test clients: 8 base rate: 0.517
Frozen rule precision@50: 0.72
Tree precision@50: 0.56
Confusion matrix, threshold .5: [[774, 2666], [433, 3242]]
Feature importance: {'impressions_90d': np.float64(0.553), 'days_since_last_update': np.float64(0.0), 'avg_position': np.float64(0.091), 'ctr': np.float64(0.078), 'content_age_days': np.float64(0.249), 'missingindicator_avg_position': np.float64(0.029)}


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

| Held-out comparison | Precision@50 |
|---|---:|
| Frozen stale-and-visible rule | 0.72 |
| Depth-3 tree | 0.56 |
| Held-out decline base rate | 0.517 |

The test contains 7,115 items across eight clients. At the 0.5 classification threshold, the confusion matrix is [[774, 2666], [433, 3242]]: many non-declining items were flagged. Tree impurity importance emphasizes impressions (0.553) and content age (0.249), which describe this fit, not causality or reliable importance under correlated inputs. The rule wins this particular top-50 comparison; ties and a single client split make the exact values unstable. This snapshot's features overlap the target period, so neither result demonstrates forecasting. A time-separated warehouse test and repeated client splits are needed before any operational use.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.